# Exact 19-nt spacer comparison: GRCh38 vs. K562 hap2 MHC

Run this notebook from the root of the repository — every path below is
repo-relative. The one external input is a GRCh38 FASTA, which is too large to
track here; set `GRCH38_FA` to wherever you keep it.

The `*.tsv.gz` result files committed at the repo root are the outputs of a
previous run, so the analysis cells at the end work without re-running the
alignment pipeline (which needs `bowtie2` and a GRCh38 index).


In [ ]:
# The pipeline script below builds both bowtie2 indexes itself and downloads
# the K562 hap2 MHC FASTA, so this cell is only needed to build the K562 index
# by hand.
!bowtie2-build analysis_trim1bp/reference/K562.hap2.MHC.fa analysis_trim1bp/index/k562_mhc


`spacer.py` is tracked in the repo — a quick one-off that dumps the `spacer`
column of the IGVF guide table to FASTA. It defaults to
`IGVFFI2404DYFG.tsv.gz` -> `spacers.fa` at the repo root, and both are
committed, so the next cell just re-generates `spacers.fa`.

The full pipeline does not use this script; it uses
`extract_spacers_trim1bp.py`, which validates and trims the spacers.


In [ ]:
!python3 spacer.py


In [ ]:
%%bash
GRCH38_FA=/path/to/hg38.fa

THREADS=16 bash exact_spacer_mhc_analysis_trim1bp.sh \
  IGVFFI2404DYFG.tsv.gz \
  "$GRCH38_FA" \
  analysis_trim1bp \
  2>&1 | tee trim1bp_pipeline.log


In [ ]:
# The individual alignment step the pipeline runs, shown for reference.
!bowtie2 \
  --end-to-end \
  -a \
  --no-unal \
  -f \
  -p 8 \
  -x analysis_trim1bp/index/grch38 \
  -U analysis_trim1bp/tmp/unique_trimmed_19nt_spacers.fa \
  -S analysis_trim1bp/sam/grch38_trim1bp.sam \
  2> bowtie2_error.log


In [ ]:
# Both analyze_*.py scripts default to the data files alongside them and
# write into results/, which is gitignored. Override with --summary / --outdir.
!python3 analyze_k562_losses.py


In [ ]:
!python3 analyze_intended_MHC_k562_losses.py
